# Project 1 — Issue Report Classification
## Notebook 01: Data loading, persistence and exploratory data analysis

**Track A deliverable.** Artificial Intelligence for Software Engineering, 2026–2027.

This notebook covers three things:

1. **Loading** the official NLBSE'24 dataset (3,000 labelled issue reports, split 50/50).
2. **Persistence** — demonstrating that the in-memory and file-based layers are interchangeable, which is an explicit non-functional requirement of the project.
3. **EDA** — the dataset characterisation that goes into the *Dataset* section of the report.

All analysis calls only the abstract `IssueRepository` interface, so nothing below depends on where the data physically lives.

In [ ]:
from pathlib import Path

# The package is installed in editable mode by the dev container
# (`pip install -e ".[dev]"`), so `import ai4se` works from any directory.
from ai4se.loader import PROJECT_ROOT

%matplotlib inline
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")

import pandas as pd
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 80)

from ai4se import eda
from ai4se.loader import load_dataset, load_split
from ai4se.model import LABELS, REPOSITORIES
from ai4se.preprocessing import clean_text, make_cleaner
from ai4se.repository import make_repository

print("project root:", PROJECT_ROOT)
print("labels      :", LABELS)
print("repositories:", REPOSITORIES)

---
## 1. Loading the dataset

The competition publishes two balanced CSV files. `load_split` downloads them once into `data/raw/` and returns a repository. Note the `kind` argument — that single string is the entire persistence switch.

In [ ]:
train = load_split("train", kind="memory")
test  = load_split("test",  kind="memory")

print(train)
print(test)

In [ ]:
# What one record looks like.
issue = train[0]
print("repo      :", issue.repo)
print("created   :", issue.created_at)
print("label     :", issue.label)
print("title     :", issue.title)
print("words     :", issue.word_count)
print("body[:300]:")
print(issue.body[:300])

---
## 2. The persistence layer

The requirement from the project specification:

> The data must be managed both in memory and through files present in the file system. The application must be designed in such a way that the use of one of the two persistence layers involves few changes to the application itself.

Below, the *same* analysis function is applied to both layers and returns identical results. The function is typed against the abstract interface and cannot tell them apart.

In [ ]:
def summarise(repository):
    """Business logic: only calls methods declared on IssueRepository."""
    return {
        "n": len(repository),
        "projects": len(repository.repos()),
        "distribution": repository.label_distribution(),
        "react_issues": len(repository.by_repo("facebook/react")),
    }

in_memory  = load_split("train", kind="memory")
from_file  = load_split("train", kind="file")

print(f"{type(in_memory).__name__:<28}", summarise(in_memory))
print(f"{type(from_file).__name__:<28}", summarise(from_file))
print()
print("identical:", summarise(in_memory) == summarise(from_file))

In [ ]:
# Round-trip through both file formats, to show the entity survives serialisation.
scratch = PROJECT_ROOT / "data" / "processed"

for fmt in ("csv", "json"):
    path = scratch / f"roundtrip_demo.{fmt}"
    written = make_repository("file", issues=train.all()[:50], path=path)
    written.save()
    reloaded = make_repository("file", path=path)
    ok = [i.to_dict() for i in reloaded] == [i.to_dict() for i in train.all()[:50]]
    print(f"{fmt:>5}: {len(reloaded)} issues reloaded, byte-identical entities: {ok}")
    path.unlink()

---
## 3. Exploratory data analysis

### 3.1 Overview and integrity checks

In [ ]:
eda.overview(train)

In [ ]:
eda.overview(test)

### 3.2 Class balance

The organisers balanced the dataset deliberately: 100 issues per (project, class) cell in each split. This has two consequences to state in the report:

- micro- and macro-averaged F1 coincide, so the choice of averaging is not a source of disagreement here;
- accuracy is a meaningful metric for once, though F1 remains the ranking metric of the competition.

In [ ]:
eda.label_distribution(train)

In [ ]:
fig = eda.plot_label_distribution(train)
eda.save_figure(fig, PROJECT_ROOT / "results" / "figures" / "label_distribution.png")

### 3.3 Issue length

The distribution is extremely skewed: the median issue is around 150 words but the maximum exceeds 21,000. This is the empirical justification for truncating the input, and for reporting results with and without truncation in the ablation.

In [ ]:
eda.length_statistics(train)

In [ ]:
fig = eda.plot_length_distribution(train)
eda.save_figure(fig, PROJECT_ROOT / "results" / "figures" / "length_distribution.png")

### 3.4 How much of the text is not natural language?

Issue bodies are Markdown. Before deciding what to strip, measure what is actually there. The table below is the percentage of issues in each class containing each kind of structural noise.

In [ ]:
eda.structural_noise(train)

Two observations worth a sentence each in the report:

- Code blocks and stack traces are far more common in **bug** reports than in feature requests. Stripping them removes noise but also removes a genuine signal, so the ablation must check whether cleaning actually helps.
- Roughly 40% of issues use a GitHub issue template, whose headings (`### Steps to reproduce`, `### Expected behaviour`) are boilerplate that appears identically across classes within a project.

### 3.5 Effect of the cleaning pipeline

Three levels are available. `raw` is the control condition, `light` removes structural noise only (appropriate for transformer models, which handle casing and stop words themselves), `full` adds lowercasing, punctuation removal, stop-word removal and lemmatisation (appropriate for TF-IDF).

> **Result from Track B (notebook 03).** The ablation later showed that `full`
> cleaning is the *worst* of the three levels for TF-IDF models, costing about
> 0.015 macro F1 against `light`. Stop-word removal deletes *would*, *could*
> and *please* — exactly the modal words that section 3.6 below identifies as
> the signature of a feature request. The project therefore settled on `light`
> (`ai4se.classical.TUNED_PREPROCESSING`). The `full` outputs are still written
> here so the ablation can be reproduced.

In [ ]:
example = train.by_label("bug")[3]

for level in ("raw", "light", "full"):
    text = clean_text(example.raw_text, level=level)
    print(f"--- {level} ({len(text.split())} words) " + "-" * 40)
    print(text[:400])
    print()

In [ ]:
eda.cleaning_impact(train, sample=500)

### 3.6 Are the classes separable with a bag of words?

Before training anything, check that class-characteristic vocabulary exists. Terms are ranked by **document frequency** — the number of issues containing them — rather than raw token count, because a few enormous issues would otherwise fill the table with vocabulary from a single log dump.

In [ ]:
train_clean = load_split("train", kind="memory")
train_clean.apply(make_cleaner(level="full", max_words=400))

eda.distinctive_terms(train_clean, n=10)

In [ ]:
for label in LABELS:
    print(f"--- {label} " + "-" * 40)
    print(eda.top_terms(train_clean, label, n=12).to_string(index=False))
    print()

The feature-request column is the informative one: *allow*, *nice*, *considered*, *easily*, *approach* are modal and evaluative words rather than technical terms, which is exactly the kind of signal a lexical model can exploit. The bug and question columns are dominated by project-specific technical vocabulary, which suggests **per-project classifiers will beat a single global one** — and that is in fact what the competition requires.

### 3.7 Per-project view

The competition asks for one classifier per project, evaluated separately, then averaged across the five. The SetFit baseline scores range from 0.7555 on `bitcoin/bitcoin` to 0.8718 on `facebook/react`, so project difficulty varies substantially and should be reported separately.

In [ ]:
rows = []
for repo in REPOSITORIES:
    issues = train_clean.by_repo(repo)
    vocabulary = {term for i in issues for term in i.text.split()}
    rows.append({
        "repository": repo,
        "issues": len(issues),
        "mean_words_raw": round(sum(i.word_count for i in issues) / len(issues), 1),
        "mean_words_clean": round(sum(len(i.text.split()) for i in issues) / len(issues), 1),
        "vocabulary": len(vocabulary),
    })
pd.DataFrame(rows)

---
## 4. Persisting the cleaned dataset for the modelling tracks

This is the handover point. Tracks B (classical ML), C (deep learning) and D (baseline and evaluation) start from these files and never re-run the cleaning themselves, which guarantees that all four sets of results are computed on identical inputs.

In [ ]:
processed = PROJECT_ROOT / "data" / "processed"
processed.mkdir(parents=True, exist_ok=True)

for split in ("train", "test"):
    for level, max_words in [("full", 400), ("light", 512)]:
        repository = load_split(split, kind="memory")
        repository.apply(make_cleaner(level=level, max_words=max_words))

        destination = processed / f"issues_{split}_{level}.csv"
        make_repository("file", issues=repository.all(), path=destination).save()
        print(f"{destination.name:<34} {len(repository):>5} issues")

In [ ]:
# Verification: reload one of the written files through the file layer and
# confirm the business logic sees exactly what it saw in memory.
check = make_repository("file", path=processed / "issues_train_full.csv")
print(check)
print(check.label_distribution())
print()
print("sample cleaned text:")
print(check[0].text[:250])

---
## 5. Summary of findings for the report

| Finding | Value | Consequence |
|---|---|---|
| Dataset size | 1,500 train / 1,500 test | Small; favours few-shot and pretrained approaches over training from scratch |
| Balance | 100 issues per (project, class) cell | Micro- and macro-F1 coincide; no resampling needed (and it is forbidden on the test set) |
| Label cardinality | Exactly one label per issue | **Multi-class, not multi-label** — multi-labelled issues were excluded by the organisers |
| Median length | ~150 words, max >21,000 | Truncation required; report the threshold as a hyperparameter |
| Code blocks | 47% of issues, 58% of bugs | Cleaning removes signal as well as noise — confirmed in notebook 03, where `full` cleaning *loses* 0.015 macro F1 |
| Duplicates | 1 in train | Negligible, but document the check |
| Vocabulary overlap across projects | Low | Per-project classifiers are justified, as the competition requires |

**Open question for the lecturer:** the project brief describes the task as multi-label classification, but the NLBSE'24 dataset is single-label multi-class — issues carrying more than one label were removed by the organisers. Confirm whether the intent was Project 3 (Jigsaw toxic comments), which genuinely is multi-label.